# 🏥 ALDIMI Predict — Notebook de Análisis y Modelado
## Machine Learning 1ACC0057 | Hito 2 — Trabajo Parcial
**Universidad Peruana de Ciencias Aplicadas | Abril 2026**

Este notebook contiene:
- **Frente 1:** EDA y modelado baseline para predicción de inventario (7 y 14 días)
- **Frente 2:** EDA y modelado baseline para clasificación de riesgo de pacientes (Bajo/Medio/Alto)


## 0. Importaciones y configuración

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, label_binarize
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, roc_auc_score,
                              roc_curve, auc, classification_report)

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

# Rutas de datos
PATH = 'data_nueva/'
print("✓ Librerías cargadas")


---
## 📦 FRENTE 1: Predicción de Inventario
### Fase 2 — Comprensión de los datos (EDA)


### 1.1 Carga y exploración inicial de datos

In [ ]:
# Dataset anual (para EDA de categorías y calidad)
df_orig = pd.read_csv(PATH + 'aldimi_dataset_completo.csv')

# Dataset semanal (para modelado)
df_sem = pd.read_csv(PATH + 'aldimi_dataset_semanal.csv')

print(f"Dataset completo: {df_orig.shape}")
print(f"Dataset semanal:  {df_sem.shape}")
print()
print("Columnas dataset semanal:", list(df_sem.columns))
df_orig.head(3)


### 1.2 Calidad de datos — Valores nulos y tipos

In [ ]:
print("=== CALIDAD DE DATOS — Dataset Completo ===")
print(f"Valores nulos por columna:")
print(df_orig.isnull().sum())
print(f"\nExistencias negativas: {(df_orig['existencias_actuales'] < 0).sum()}")
print(f"Artículos con tasa_rotacion > 1 (stock agotado): {(df_orig['tasa_rotacion'] > 1).sum()}")
print()
print("=== Distribución del target ===")
print(df_orig['necesita_reposicion'].value_counts())
print(f"% necesita reposición: {df_orig['necesita_reposicion'].mean()*100:.1f}%")


### 1.3 Gráfico 1 — Distribución de artículos por categoría

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
cats = df_orig['categoria'].value_counts()
colors = plt.cm.Blues(np.linspace(0.4, 0.9, len(cats)))
bars = ax.barh(cats.index[::-1], cats.values[::-1], color=colors[::-1], edgecolor='white')
for bar, v in zip(bars, cats.values[::-1]):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            str(v), va='center', fontsize=9)
ax.set_xlabel('N° de artículos', fontsize=11)
ax.set_title('Distribución de artículos por categoría\n(Almacén ALDIMI)', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('g1_categorias.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Gráfico guardado: g1_categorias.png")


### 1.4 Gráfico 2 — Balance de clases (7 y 14 días)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (col, label) in zip(axes, [('alerta_7_dias','Alerta 7 días'), ('alerta_14_dias','Alerta 14 días')]):
    vc  = df_sem[col].value_counts().sort_index()
    pct = vc / vc.sum() * 100
    bars = ax.bar(['Sin alerta (0)', 'Con alerta (1)'], vc.values,
                  color=['#4CAF50', '#F44336'], edgecolor='white', width=0.5)
    for b, p in zip(bars, pct):
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 30,
                f'{p:.1f}%', ha='center', fontweight='bold', fontsize=11)
    ax.set_title(f'Balance de clases\n{label}', fontweight='bold')
    ax.set_ylabel('N° de registros')
    ax.set_ylim(0, max(vc.values) * 1.18)

plt.suptitle('Frente 1 — Balance del target de inventario', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('g2_balance_inventario.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Gráfico guardado: g2_balance_inventario.png")


### 1.5 Gráfico 3 — Top 15 artículos por tasa de rotación

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
top15 = df_orig.nlargest(15, 'tasa_rotacion')[['codigo_articulo','tasa_rotacion']].reset_index(drop=True)
colors = ['#D32F2F' if t > 1 else '#FF9800' if t > 0.5 else '#4CAF50' for t in top15['tasa_rotacion']]
ax.barh(top15['codigo_articulo'][::-1], top15['tasa_rotacion'][::-1], color=colors[::-1])
ax.axvline(1.0, color='red', linestyle='--', alpha=0.8, label='Tasa = 1.0 (stock agotado)')
ax.set_xlabel('Tasa de rotación (salidas / stock disponible)', fontsize=11)
ax.set_title('Top 15 artículos con mayor tasa de rotación\n(rojo = agotado | naranja = riesgo alto)',
             fontweight='bold', fontsize=13)
ax.legend()
plt.tight_layout()
plt.savefig('g3_rotacion.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Gráfico guardado: g3_rotacion.png")


### 1.6 Gráfico 4 — Mapa de calor de correlaciones

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
num_cols = ['existencias_iniciales','total_entradas','total_salidas',
            'existencias_actuales','tasa_rotacion',
            'frecuencia_salidas','frecuencia_entradas','necesita_reposicion']
corr = df_orig[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, linewidths=0.5, ax=ax, annot_kws={'size': 9})
ax.set_title('Correlaciones entre variables numéricas\n(Frente 1 — Inventario)', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('g4_correlaciones_inv.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Gráfico guardado: g4_correlaciones_inv.png")


### 1.7 Gráfico 5 — Distribución de existencias actuales

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))

# Boxplot completo
df_orig['existencias_actuales'].plot(kind='box', ax=axes[0],
    patch_artist=True, boxprops=dict(facecolor='#BBDEFB', color='#1565C0'),
    medianprops=dict(color='#1565C0', linewidth=2))
axes[0].set_title('Distribución existencias actuales\n(con outliers)', fontweight='bold')
axes[0].set_ylabel('Unidades')

# Histograma sin outliers extremos
df_filt = df_orig[(df_orig['existencias_actuales'] >= -20) & (df_orig['existencias_actuales'] <= 50)]
axes[1].hist(df_filt['existencias_actuales'], bins=35, color='#1976D2', edgecolor='white')
axes[1].axvline(0, color='red', linestyle='--', label='Stock = 0')
axes[1].axvline(5, color='orange', linestyle='--', label='Umbral alerta (5 uds)')
axes[1].set_title('Histograma existencias [-20, 50]\n(sin outliers extremos)', fontweight='bold')
axes[1].set_xlabel('Unidades')
axes[1].legend()

plt.tight_layout()
plt.savefig('g5_existencias.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Gráfico guardado: g5_existencias.png")


---
### Fase 3 — Preparación de datos (Frente 1)
### 1.8 Preprocesamiento del dataset semanal

In [ ]:
# Encoding y escalado
le_cat = LabelEncoder()
df_sem['cat_enc'] = le_cat.fit_transform(df_sem['categoria'])

features_inv = ['cat_enc','ocupacion_albergue','stock_inicio_semana',
                'ingresos_semana','salidas_semana','rolling_avg_salidas_3sem','semana_del_año']

scaler_inv = MinMaxScaler()
X_inv = scaler_inv.fit_transform(df_sem[features_inv])

print(f"Features: {features_inv}")
print(f"Shape X: {X_inv.shape}")
print(f"Categorías codificadas: {list(le_cat.classes_)}")


### Fase 4 — Modelado Baseline (Frente 1)

### 1.9 Entrenamiento y métricas — Alerta 7 días

In [ ]:
y7 = df_sem['alerta_7_dias'].values
X_tr7, X_te7, y_tr7, y_te7 = train_test_split(X_inv, y7, test_size=0.30, random_state=42, stratify=y7)

modelos_f1 = {
    'Naive Bayes':     GaussianNB(),
    'KNN (k=5)':       KNeighborsClassifier(n_neighbors=5),
    'Árbol (depth=5)': DecisionTreeClassifier(max_depth=5, random_state=42, class_weight='balanced'),
}

print(f"{'Modelo':22s} | Accuracy | Precisión | Recall  | F1-Score | AUC-ROC")
print("-" * 75)
resultados_7 = {}
for nombre, modelo in modelos_f1.items():
    modelo.fit(X_tr7, y_tr7)
    y_pred = modelo.predict(X_te7)
    y_prob = modelo.predict_proba(X_te7)[:, 1]
    acc  = accuracy_score(y_te7, y_pred)
    prec = precision_score(y_te7, y_pred)
    rec  = recall_score(y_te7, y_pred)
    f1   = f1_score(y_te7, y_pred)
    auc_ = roc_auc_score(y_te7, y_prob)
    resultados_7[nombre] = dict(acc=acc, prec=prec, rec=rec, f1=f1, auc=auc_)
    print(f"{nombre:22s} | {acc:.4f}   | {prec:.4f}    | {rec:.4f}  | {f1:.4f}   | {auc_:.4f}")


### 1.10 Gráfico 6 — Matrices de confusión (7 días)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (nombre, modelo) in zip(axes, modelos_f1.items()):
    y_pred = modelo.predict(X_te7)
    cm = confusion_matrix(y_te7, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, cbar=False,
                xticklabels=['Sin alerta','Con alerta'],
                yticklabels=['Sin alerta','Con alerta'])
    ax.set_title(f'{nombre}\nF1={resultados_7[nombre]["f1"]:.3f}', fontweight='bold')
    ax.set_xlabel('Predicho')
    ax.set_ylabel('Real')

plt.suptitle('Matrices de confusión — Frente 1 (Alerta 7 días)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('g9_cm_f1_7dias.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Gráfico guardado: g9_cm_f1_7dias.png")


### 1.11 Gráfico 7 — Curvas ROC (7 días)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
colores = ['#1976D2', '#E65100', '#2E7D32']
for (nombre, modelo), color in zip(modelos_f1.items(), colores):
    y_prob = modelo.predict_proba(X_te7)[:, 1]
    fpr, tpr, _ = roc_curve(y_te7, y_prob)
    auc_val = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{nombre} (AUC={auc_val:.3f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Clasificador aleatorio')
ax.set_xlabel('Tasa de Falsos Positivos', fontsize=11)
ax.set_ylabel('Tasa de Verdaderos Positivos', fontsize=11)
ax.set_title('Curvas ROC — Frente 1 (Alerta 7 días)', fontweight='bold', fontsize=13)
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('g10_roc_f1.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Gráfico guardado: g10_roc_f1.png")


### 1.12 Métricas — Alerta 14 días

In [ ]:
y14 = df_sem['alerta_14_dias'].values
X_tr14, X_te14, y_tr14, y_te14 = train_test_split(X_inv, y14, test_size=0.30, random_state=42, stratify=y14)

modelos_f1_14 = {
    'Naive Bayes':     GaussianNB(),
    'KNN (k=5)':       KNeighborsClassifier(n_neighbors=5),
    'Árbol (depth=5)': DecisionTreeClassifier(max_depth=5, random_state=42, class_weight='balanced'),
}

print(f"{'Modelo':22s} | Accuracy | Precisión | Recall  | F1-Score | AUC-ROC")
print("-" * 75)
for nombre, modelo in modelos_f1_14.items():
    modelo.fit(X_tr14, y_tr14)
    y_pred = modelo.predict(X_te14)
    y_prob = modelo.predict_proba(X_te14)[:, 1]
    acc  = accuracy_score(y_te14, y_pred)
    prec = precision_score(y_te14, y_pred)
    rec  = recall_score(y_te14, y_pred)
    f1   = f1_score(y_te14, y_pred)
    auc_ = roc_auc_score(y_te14, y_prob)
    print(f"{nombre:22s} | {acc:.4f}   | {prec:.4f}    | {rec:.4f}  | {f1:.4f}   | {auc_:.4f}")


---
## 👤 FRENTE 2: Clasificación de Riesgo de Pacientes
### Fase 2 — Comprensión de los datos (EDA)


### 2.1 Carga y exploración del dataset de pacientes

In [ ]:
df_pac = pd.read_csv(PATH + 'aldimi_pacientes_sintetico.csv')
print(f"Shape: {df_pac.shape}")
print(f"Columnas: {list(df_pac.columns)}")
print()
print("Balance de clases:")
vc = df_pac['nivel_riesgo'].value_counts()[['Bajo','Medio','Alto']]
for k, v in vc.items():
    print(f"  {k}: {v} ({v/len(df_pac)*100:.1f}%)")
df_pac.head(3)


### 2.2 Gráfico 8 — Balance de clases (pacientes)

In [ ]:
PALETTE = {'Bajo':'#4CAF50','Medio':'#FF9800','Alto':'#F44336'}
fig, ax = plt.subplots(figsize=(7, 4))
vc = df_pac['nivel_riesgo'].value_counts()[['Bajo','Medio','Alto']]
bars = ax.bar(vc.index, vc.values, color=[PALETTE[k] for k in vc.index],
              edgecolor='white', width=0.5)
for b, v in zip(bars, vc.values):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 3,
            f'{v}\n({v/len(df_pac)*100:.1f}%)', ha='center', fontweight='bold')
ax.set_title('Balance de clases — nivel_riesgo\n(Frente 2: Pacientes ALDIMI)', fontweight='bold', fontsize=13)
ax.set_ylabel('N° de pacientes')
ax.set_ylim(0, max(vc.values) * 1.22)
plt.tight_layout()
plt.savefig('g6_balance_pacientes.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Gráfico guardado: g6_balance_pacientes.png")


### 2.3 Gráfico 9 — Nivel de riesgo por etapa del cáncer

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ct = pd.crosstab(df_pac['etapa_cancer'], df_pac['nivel_riesgo'],
                 normalize='index')[['Bajo','Medio','Alto']] * 100
ct.plot(kind='bar', ax=ax, color=[PALETTE[k] for k in ['Bajo','Medio','Alto']],
        edgecolor='white', width=0.7)
ax.set_xlabel('Etapa del cáncer', fontsize=11)
ax.set_ylabel('% de pacientes', fontsize=11)
ax.set_title('Nivel de riesgo por etapa del cáncer\n(Frente 2 — Pacientes)', fontweight='bold', fontsize=13)
ax.legend(title='Nivel de riesgo')
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('g7_etapa_riesgo.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Gráfico guardado: g7_etapa_riesgo.png")


### 2.4 Gráfico 10 — Mapa de calor de correlaciones (pacientes)

In [ ]:
ord_maps = {
    'etapa_cancer':           {'I':0,'II':1,'III':2,'IV':3},
    'estado_fisico':          {'Caminando':0,'Con ayuda parcial':1,'Permanece en cama':2,'Usa silla de ruedas':3},
    'estado_nutricional':     {'Normal':0,'Desnutrición leve':1,'Desnutrición severa':2},
    'adherencia_tratamiento': {'Alta':0,'Media':1,'Baja':2},
    'apoyo_familiar':         {'Fuerte':0,'Moderado':1,'Limitado':2},
    'acceso_medicamentos':    {'Completo':0,'Parcial':1,'Limitado':2},
}
df_corr = df_pac.copy()
for col, mp in ord_maps.items():
    df_corr[col] = df_corr[col].map(mp)
df_corr['nivel_riesgo_num'] = df_pac['nivel_riesgo'].map({'Bajo':0,'Medio':1,'Alto':2})

num_p = list(ord_maps.keys()) + ['edad','num_reingresos','presencia_infeccion',
                                  'frecuencia_hospitalizacion_3m','perdida_peso_reciente',
                                  'num_comorbilidades','nivel_riesgo_num']
fig, ax = plt.subplots(figsize=(12, 9))
mask_p = np.triu(np.ones((len(num_p), len(num_p)), dtype=bool))
sns.heatmap(df_corr[num_p].corr(), mask=mask_p, annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, linewidths=0.4, ax=ax, annot_kws={'size':7})
ax.set_title('Correlaciones entre variables\n(Frente 2 — Pacientes)', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('g8_correlaciones_pac.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Gráfico guardado: g8_correlaciones_pac.png")


### Fase 3 — Preparación de datos (Frente 2)
### 2.5 Preprocesamiento

In [ ]:
# Encoding ordinal
ord_maps_full = {
    'etapa_cancer':               {'I':0,'II':1,'III':2,'IV':3},
    'estado_fisico':              {'Caminando':0,'Con ayuda parcial':1,'Permanece en cama':2,'Usa silla de ruedas':3},
    'estado_nutricional':         {'Normal':0,'Desnutrición leve':1,'Desnutrición severa':2},
    'adherencia_tratamiento':     {'Alta':0,'Media':1,'Baja':2},
    'apoyo_familiar':             {'Fuerte':0,'Moderado':1,'Limitado':2},
    'acceso_medicamentos':        {'Completo':0,'Parcial':1,'Limitado':2},
    'estado_emocional_paciente':  {'Estable':0,'Ansioso':1,'Deprimido':2},
    'grado_instruccion_cuidador': {'Sin estudios':0,'Primaria':1,'Secundaria':2,
                                   'Superior técnica':3,'Superior universitaria':4},
}
cat_ohe = ['sexo','diagnostico','tipo_tratamiento','motivo_ingreso','motivo_reingreso','lugar_procedencia']

df_enc = pd.get_dummies(df_pac, columns=cat_ohe, drop_first=True)
for col, mp in ord_maps_full.items():
    df_enc[col+'_enc'] = df_pac[col].map(mp)
df_enc['estado_emoc_enc'] = df_pac['estado_emocional_paciente'].map({'Estable':0,'Ansioso':1,'Deprimido':2})

feat_cols = ([c+'_enc' for c in ord_maps_full] + ['estado_emoc_enc',
             'edad','distancia_origen_km','meses_en_tratamiento','num_reingresos',
             'presencia_infeccion','frecuencia_hospitalizacion_3m',
             'perdida_peso_reciente','num_comorbilidades'] +
             [c for c in df_enc.columns if any(c.startswith(b+'_') for b in cat_ohe)])

scaler_p = MinMaxScaler()
X_p = scaler_p.fit_transform(df_enc[feat_cols].values.astype(float))

le_target = LabelEncoder()
y_p = le_target.fit_transform(df_pac['nivel_riesgo'])
clases = le_target.classes_

X_tr_p, X_te_p, y_tr_p, y_te_p = train_test_split(X_p, y_p, test_size=0.30, random_state=42, stratify=y_p)

print(f"Clases: {clases}")
print(f"Train: {len(X_tr_p)} | Test: {len(X_te_p)}")
print(f"Total features: {len(feat_cols)}")


### Fase 4 — Modelado Baseline (Frente 2)
### 2.6 Entrenamiento y métricas

In [ ]:
modelos_f2 = {
    'Naive Bayes':     GaussianNB(),
    'KNN (k=5)':       KNeighborsClassifier(n_neighbors=5),
    'Árbol (depth=5)': DecisionTreeClassifier(max_depth=5, random_state=42, class_weight='balanced'),
}
y_bin = label_binarize(y_te_p, classes=[0, 1, 2])

print(f"{'Modelo':22s} | Accuracy | F1-Macro | F1-Weighted | AUC-Macro")
print("-" * 72)
resultados_f2 = {}
for nombre, modelo in modelos_f2.items():
    modelo.fit(X_tr_p, y_tr_p)
    y_pred = modelo.predict(X_te_p)
    y_prob = modelo.predict_proba(X_te_p)
    acc  = accuracy_score(y_te_p, y_pred)
    f1m  = f1_score(y_te_p, y_pred, average='macro', zero_division=0)
    f1w  = f1_score(y_te_p, y_pred, average='weighted', zero_division=0)
    auc_ = roc_auc_score(y_bin, y_prob, multi_class='ovr', average='macro')
    resultados_f2[nombre] = dict(acc=acc, f1m=f1m, f1w=f1w, auc=auc_)
    print(f"{nombre:22s} | {acc:.4f}   | {f1m:.4f}   | {f1w:.4f}      | {auc_:.4f}")


### 2.7 Gráfico 11 — Matrices de confusión (pacientes)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (nombre, modelo) in zip(axes, modelos_f2.items()):
    y_pred = modelo.predict(X_te_p)
    cm = confusion_matrix(y_te_p, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges', ax=ax, cbar=False,
                xticklabels=clases, yticklabels=clases)
    ax.set_title(f'{nombre}\nF1-macro={resultados_f2[nombre]["f1m"]:.3f}', fontweight='bold')
    ax.set_xlabel('Predicho')
    ax.set_ylabel('Real')

plt.suptitle('Matrices de confusión — Frente 2 (Nivel de riesgo: Bajo/Medio/Alto)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('g11_cm_f2.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Gráfico guardado: g11_cm_f2.png")


### 2.8 Gráfico 12 — Curvas ROC por clase (pacientes, OvR)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
color_sets = [['#1976D2','#E65100','#2E7D32'],
              ['#7B1FA2','#00838F','#F57F17'],
              ['#C62828','#2E7D32','#1565C0']]

for ax, (nombre, modelo), colors in zip(axes, modelos_f2.items(), color_sets):
    y_prob = modelo.predict_proba(X_te_p)
    for i, (cls, col) in enumerate(zip(clases, colors)):
        fpr, tpr, _ = roc_curve(y_bin[:, i], y_prob[:, i])
        auc_val = auc(fpr, tpr)
        ax.plot(fpr, tpr, color=col, lw=2, label=f'{cls} (AUC={auc_val:.3f})')
    ax.plot([0,1],[0,1],'k--', lw=1)
    ax.set_title(nombre, fontweight='bold')
    ax.set_xlabel('FPR')
    ax.set_ylabel('TPR')
    ax.legend(fontsize=8)

plt.suptitle('Curvas ROC — Frente 2 (Nivel de riesgo, One-vs-Rest)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('g12_roc_f2.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Gráfico guardado: g12_roc_f2.png")


---
## ✅ Resumen final de métricas

In [ ]:
print("=" * 65)
print("FRENTE 1 — INVENTARIO")
print("=" * 65)
print(f"\n{'Modelo':22s} | {'F1 (7d)':8s} | {'AUC (7d)':9s} | {'F1 (14d)':9s} | {'AUC (14d)'}")
print("-" * 65)
nombres = ['Naive Bayes','KNN (k=5)','Árbol (depth=5)']
f1_7  = [0.842, 0.849, 0.970]
auc_7 = [0.796, 0.833, 0.981]
f1_14 = [0.863, 0.846, 0.944]
auc_14= [0.772, 0.806, 0.975]
for n, a, b, c, d in zip(nombres, f1_7, auc_7, f1_14, auc_14):
    print(f"{n:22s} | {a:.3f}    | {b:.3f}     | {c:.3f}     | {d:.3f}")

print()
print("=" * 65)
print("FRENTE 2 — PACIENTES")
print("=" * 65)
print(f"\n{'Modelo':22s} | {'Accuracy':9s} | {'F1-Macro':9s} | {'AUC-Macro'}")
print("-" * 55)
for nombre, v in resultados_f2.items():
    print(f"{nombre:22s} | {v['acc']:.3f}     | {v['f1m']:.3f}     | {v['auc']:.3f}")

print()
print("Modelos seleccionados para Hito 3:")
print("  Frente 1: Árbol de Decisión (depth=5) — F1=0.970 / AUC=0.981")
print("  Frente 2: Árbol de Decisión (depth=5) — mejor F1-Macro")
print("  Siguiente paso: XGBoost + Random Forest con tuning de hiperparámetros")


---
## 🚀 HITO 3 — Modelado Avanzado e Interpretabilidad
### Fases 4 (avanzada) y 5 de CRISP-DM
> Random Forest + XGBoost con ajuste de hiperparámetros · Validación cruzada · SHAP


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, cross_val_score
from sklearn.metrics import ConfusionMatrixDisplay
from scipy.stats import randint, uniform
import xgboost as xgb
import shap
import joblib, os

shap.initjs()
print("✓ Librerías Hito 3 cargadas")
print(f"  XGBoost: {xgb.__version__}  |  SHAP: {shap.__version__}")


---
## 📦 FRENTE 1 — Inventario (Hito 3)
### 3.1 Random Forest con RandomizedSearchCV — Alerta 7 días


In [ ]:
# ── Espacio de hiperparámetros ─────────────────────────────────────────
param_rf_inv = {
    'n_estimators':      randint(100, 500),
    'max_depth':         [None, 5, 8, 12, 16],
    'min_samples_split': randint(2, 12),
    'min_samples_leaf':  randint(1, 8),
    'max_features':      ['sqrt', 'log2', 0.5],
    'class_weight':      ['balanced', 'balanced_subsample'],
}

cv_inv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

search_rf_inv = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_distributions=param_rf_inv,
    n_iter=50, scoring='f1', cv=cv_inv,
    random_state=42, n_jobs=-1, verbose=0
)
search_rf_inv.fit(X_tr7, y_tr7)

rf_inv_best = search_rf_inv.best_estimator_
print("✓ Random Forest — Inventario 7d")
print(f"  Mejores hiperparámetros: {search_rf_inv.best_params_}")
print(f"  F1 CV (train): {search_rf_inv.best_score_:.4f}")


In [ ]:
# ── XGBoost con RandomizedSearchCV ────────────────────────────────────
param_xgb_inv = {
    'n_estimators':    randint(100, 400),
    'max_depth':       randint(3, 9),
    'learning_rate':   uniform(0.02, 0.18),
    'subsample':       uniform(0.6, 0.4),
    'colsample_bytree':uniform(0.5, 0.5),
    'gamma':           uniform(0, 0.3),
    'reg_alpha':       uniform(0, 0.5),
    'reg_lambda':      uniform(0.5, 2.0),
}

# Proporción de clase negativa / positiva como scale_pos_weight
neg, pos = (y_tr7 == 0).sum(), (y_tr7 == 1).sum()

search_xgb_inv = RandomizedSearchCV(
    xgb.XGBClassifier(
        scale_pos_weight=neg/pos,
        eval_metric='logloss',
        random_state=42, verbosity=0, use_label_encoder=False
    ),
    param_distributions=param_xgb_inv,
    n_iter=50, scoring='f1', cv=cv_inv,
    random_state=42, n_jobs=-1, verbose=0
)
search_xgb_inv.fit(X_tr7, y_tr7)

xgb_inv_best = search_xgb_inv.best_estimator_
print("✓ XGBoost — Inventario 7d")
print(f"  Mejores hiperparámetros: {search_xgb_inv.best_params_}")
print(f"  F1 CV (train): {search_xgb_inv.best_score_:.4f}")


### 3.2 Validación cruzada estratificada — Inventario 7 días

In [ ]:
# ── Validación cruzada de los 5 modelos (baseline + avanzados) ─────────
todos_inv7 = {
    'Naive Bayes':        modelos_f1['Naive Bayes'],
    'KNN (k=5)':          modelos_f1['KNN (k=5)'],
    'Árbol (depth=5)':    modelos_f1['Árbol (depth=5)'],
    'Random Forest':      rf_inv_best,
    'XGBoost':            xgb_inv_best,
}

print(f"{'Modelo':22s} | {'F1 CV μ':>8s} | {'F1 CV σ':>8s} | {'AUC Test':>9s} | {'F1 Test':>8s}")
print("-" * 72)
resumen_inv7 = {}
for nombre, modelo in todos_inv7.items():
    cv_scores = cross_val_score(modelo, X_inv, y7, cv=cv_inv, scoring='f1', n_jobs=-1)
    y_pred = modelo.predict(X_te7)
    y_prob = modelo.predict_proba(X_te7)[:, 1]
    f1_t   = f1_score(y_te7, y_pred)
    auc_t  = roc_auc_score(y_te7, y_prob)
    resumen_inv7[nombre] = dict(cv_mu=cv_scores.mean(), cv_sd=cv_scores.std(),
                                 f1_test=f1_t, auc_test=auc_t)
    print(f"{nombre:22s} | {cv_scores.mean():.4f}   | {cv_scores.std():.4f}   | {auc_t:.4f}    | {f1_t:.4f}")


### 3.3 Gráfico 13 — Comparativa baseline vs. avanzados (Inventario 7d)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

nombres_ord = list(resumen_inv7.keys())
f1_vals   = [resumen_inv7[n]['f1_test']  for n in nombres_ord]
auc_vals  = [resumen_inv7[n]['auc_test'] for n in nombres_ord]
cv_mus    = [resumen_inv7[n]['cv_mu']    for n in nombres_ord]
cv_sds    = [resumen_inv7[n]['cv_sd']    for n in nombres_ord]

colors = ['#90A4AE','#90A4AE','#90A4AE','#1976D2','#E65100']
hatches= ['','','','','']

# Panel izquierdo: F1 Test
ax = axes[0]
bars = ax.bar(nombres_ord, f1_vals, color=colors, edgecolor='white', width=0.6)
for b, v in zip(bars, f1_vals):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.003,
            f'{v:.3f}', ha='center', fontsize=9, fontweight='bold')
ax.set_ylim(0.7, 1.02)
ax.set_ylabel('F1-Score (Test)', fontsize=11)
ax.set_title('F1 en test — Inventario 7 días
(baseline en gris · avanzados en color)', fontweight='bold')
ax.tick_params(axis='x', rotation=15)
ax.axhline(0.97, color='gray', linestyle='--', lw=1, alpha=0.6, label='Baseline mejor (0.970)')
ax.legend(fontsize=8)

# Panel derecho: F1 CV con barras de error
ax = axes[1]
ax.bar(nombres_ord, cv_mus, yerr=cv_sds, color=colors, edgecolor='white',
       width=0.6, capsize=5, error_kw=dict(elinewidth=1.2, ecolor='#333'))
for i, (mu, sd) in enumerate(zip(cv_mus, cv_sds)):
    ax.text(i, mu + sd + 0.005, f'{mu:.3f}\n±{sd:.3f}',
            ha='center', fontsize=8, fontweight='bold')
ax.set_ylim(0.7, 1.05)
ax.set_ylabel('F1 promedio (CV 5-fold)', fontsize=11)
ax.set_title('Validación cruzada (5-fold)\nInventario 7 días', fontweight='bold')
ax.tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('g13_comparativa_inv7.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Gráfico guardado: g13_comparativa_inv7.png")


### 3.4 Curvas ROC — modelos avanzados vs. baseline (Inventario 7d)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
estilos = [('--','#90A4AE'),('--','#78909C'),('--','#607D8B'),('-','#1976D2'),('-','#E65100')]
for (nombre, modelo), (ls, col) in zip(todos_inv7.items(), estilos):
    y_prob = modelo.predict_proba(X_te7)[:, 1]
    fpr, tpr, _ = roc_curve(y_te7, y_prob)
    lw = 1.5 if ls == '--' else 2.5
    ax.plot(fpr, tpr, color=col, lw=lw, ls=ls,
            label=f'{nombre} (AUC={roc_auc_score(y_te7, y_prob):.3f})')
ax.plot([0,1],[0,1],'k--',lw=1)
ax.set_xlabel('Tasa de Falsos Positivos', fontsize=11)
ax.set_ylabel('Tasa de Verdaderos Positivos', fontsize=11)
ax.set_title('Curvas ROC — Frente 1 (Inventario 7d)\nBaseline vs. Avanzados', fontweight='bold', fontsize=13)
ax.legend(loc='lower right', fontsize=9)
plt.tight_layout()
plt.savefig('g14_roc_inv7_avanzado.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Gráfico guardado: g14_roc_inv7_avanzado.png")


### 3.5 Métricas — Alerta 14 días (modelos avanzados)

In [ ]:
# ── Repetir tuning para 14d (mismo espacio de hiperparámetros) ─────────
search_rf_inv14 = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_distributions=param_rf_inv,
    n_iter=50, scoring='f1', cv=cv_inv,
    random_state=42, n_jobs=-1, verbose=0
)
search_rf_inv14.fit(X_tr14, y_tr14)
rf_inv14_best = search_rf_inv14.best_estimator_

neg14, pos14 = (y_tr14 == 0).sum(), (y_tr14 == 1).sum()
search_xgb_inv14 = RandomizedSearchCV(
    xgb.XGBClassifier(scale_pos_weight=neg14/pos14, eval_metric='logloss',
                       random_state=42, verbosity=0, use_label_encoder=False),
    param_distributions=param_xgb_inv,
    n_iter=50, scoring='f1', cv=cv_inv,
    random_state=42, n_jobs=-1, verbose=0
)
search_xgb_inv14.fit(X_tr14, y_tr14)
xgb_inv14_best = search_xgb_inv14.best_estimator_

todos_inv14 = {
    'Árbol (depth=5)': modelos_f1_14['Árbol (depth=5)'],
    'Random Forest':   rf_inv14_best,
    'XGBoost':         xgb_inv14_best,
}
print(f"{'Modelo':22s} | {'F1 Test':>8s} | {'AUC Test':>9s} | {'Precisión':>10s} | {'Recall':>7s}")
print("-" * 70)
for nombre, modelo in todos_inv14.items():
    y_pred = modelo.predict(X_te14)
    y_prob = modelo.predict_proba(X_te14)[:, 1]
    print(f"{nombre:22s} | {f1_score(y_te14,y_pred):.4f}   | {roc_auc_score(y_te14,y_prob):.4f}    | "
          f"{precision_score(y_te14,y_pred):.4f}     | {recall_score(y_te14,y_pred):.4f}")


### 3.6 Gráfico 15 — SHAP Inventario (mejor modelo)

In [ ]:
# ── Determinar mejor modelo Inventario 7d ─────────────────────────────
mejor_nombre_inv = max(resumen_inv7, key=lambda n: resumen_inv7[n]['f1_test'])
mejor_modelo_inv = todos_inv7[mejor_nombre_inv]
print(f"Mejor modelo Frente 1: {mejor_nombre_inv}")
print(f"  F1 Test = {resumen_inv7[mejor_nombre_inv]['f1_test']:.4f}")
print(f"  AUC Test = {resumen_inv7[mejor_nombre_inv]['auc_test']:.4f}")

# SHAP TreeExplainer (compatible con RF y XGBoost)
explainer_inv = shap.TreeExplainer(mejor_modelo_inv)
# Muestra del test para velocidad
np.random.seed(42)
idx_sample = np.random.choice(len(X_te7), min(200, len(X_te7)), replace=False)
shap_vals_inv = explainer_inv.shap_values(X_te7[idx_sample])

# Para clasificación binaria RF shap_values es lista [neg, pos]; tomamos clase 1
if isinstance(shap_vals_inv, list):
    sv = shap_vals_inv[1]
else:
    sv = shap_vals_inv

features_inv_labels = ['categoria','ocupacion_albergue','stock_inicio_semana',
                        'ingresos_semana','salidas_semana','rolling_avg_salidas_3sem','semana_del_año']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot (importancia media |SHAP|)
mean_shap = np.abs(sv).mean(axis=0)
order = np.argsort(mean_shap)
colors_shap = plt.cm.Blues(np.linspace(0.4, 0.9, len(features_inv_labels)))
axes[0].barh([features_inv_labels[i] for i in order], mean_shap[order],
             color=colors_shap)
axes[0].set_xlabel('Importancia media |SHAP|', fontsize=11)
axes[0].set_title(f'SHAP — Importancia de variables\n({mejor_nombre_inv}, Frente 1)',
                  fontweight='bold', fontsize=12)

# Beeswarm manual (scatter por variable)
ax2 = axes[1]
for rank, i in enumerate(order):
    vals = sv[:, i]
    feat = X_te7[idx_sample, i]
    sc = ax2.scatter(vals, np.full_like(vals, rank) + np.random.normal(0, 0.08, len(vals)),
                     c=feat, cmap='coolwarm', alpha=0.5, s=12, vmin=0, vmax=1)
ax2.set_yticks(range(len(features_inv_labels)))
ax2.set_yticklabels([features_inv_labels[i] for i in order], fontsize=9)
ax2.axvline(0, color='gray', lw=0.8, linestyle='--')
ax2.set_xlabel('Valor SHAP (impacto en predicción)', fontsize=11)
ax2.set_title('Distribución SHAP por variable\n(azul=valor bajo · rojo=valor alto)',
              fontweight='bold', fontsize=12)
plt.colorbar(sc, ax=ax2, label='Valor normalizado de la feature')

plt.tight_layout()
plt.savefig('g15_shap_inventario.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Gráfico guardado: g15_shap_inventario.png")


---
## 👤 FRENTE 2 — Pacientes (Hito 3)
### 3.7 Random Forest con RandomizedSearchCV


In [ ]:
param_rf_pac = {
    'n_estimators':      randint(100, 600),
    'max_depth':         [None, 6, 10, 15, 20],
    'min_samples_split': randint(2, 15),
    'min_samples_leaf':  randint(1, 8),
    'max_features':      ['sqrt', 'log2', 0.3, 0.5],
    'class_weight':      ['balanced', 'balanced_subsample'],
}

cv_pac = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

search_rf_pac = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_distributions=param_rf_pac,
    n_iter=60, scoring='f1_macro', cv=cv_pac,
    random_state=42, n_jobs=-1, verbose=0
)
search_rf_pac.fit(X_tr_p, y_tr_p)
rf_pac_best = search_rf_pac.best_estimator_
print("✓ Random Forest — Pacientes")
print(f"  Mejores hiperparámetros: {search_rf_pac.best_params_}")
print(f"  F1-Macro CV (train): {search_rf_pac.best_score_:.4f}")


In [ ]:
# ── XGBoost multiclase ─────────────────────────────────────────────────
param_xgb_pac = {
    'n_estimators':     randint(100, 500),
    'max_depth':        randint(3, 10),
    'learning_rate':    uniform(0.02, 0.18),
    'subsample':        uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.4, 0.6),
    'gamma':            uniform(0, 0.4),
    'reg_alpha':        uniform(0, 0.5),
    'reg_lambda':       uniform(0.5, 2.0),
}

search_xgb_pac = RandomizedSearchCV(
    xgb.XGBClassifier(
        objective='multi:softprob', num_class=3,
        eval_metric='mlogloss', random_state=42,
        verbosity=0, use_label_encoder=False
    ),
    param_distributions=param_xgb_pac,
    n_iter=60, scoring='f1_macro', cv=cv_pac,
    random_state=42, n_jobs=-1, verbose=0
)
search_xgb_pac.fit(X_tr_p, y_tr_p)
xgb_pac_best = search_xgb_pac.best_estimator_
print("✓ XGBoost — Pacientes")
print(f"  Mejores hiperparámetros: {search_xgb_pac.best_params_}")
print(f"  F1-Macro CV (train): {search_xgb_pac.best_score_:.4f}")


### 3.8 Validación cruzada estratificada — Frente 2

In [ ]:
todos_pac = {
    'Naive Bayes':        modelos_f2['Naive Bayes'],
    'KNN (k=5)':          modelos_f2['KNN (k=5)'],
    'Árbol (depth=5)':    modelos_f2['Árbol (depth=5)'],
    'Random Forest':      rf_pac_best,
    'XGBoost':            xgb_pac_best,
}

y_bin_te = label_binarize(y_te_p, classes=[0, 1, 2])

print(f"{'Modelo':22s} | {'F1-Macro CV μ':>13s} | {'F1-Macro CV σ':>13s} | {'Acc Test':>9s} | {'AUC-Macro':>9s}")
print("-" * 80)
resumen_pac = {}
for nombre, modelo in todos_pac.items():
    cv_scores = cross_val_score(modelo, X_p, y_p, cv=cv_pac, scoring='f1_macro', n_jobs=-1)
    y_pred = modelo.predict(X_te_p)
    y_prob = modelo.predict_proba(X_te_p)
    acc  = accuracy_score(y_te_p, y_pred)
    f1m  = f1_score(y_te_p, y_pred, average='macro', zero_division=0)
    auc_ = roc_auc_score(y_bin_te, y_prob, multi_class='ovr', average='macro')
    resumen_pac[nombre] = dict(cv_mu=cv_scores.mean(), cv_sd=cv_scores.std(),
                                acc=acc, f1m=f1m, auc=auc_)
    print(f"{nombre:22s} | {cv_scores.mean():.4f}        | {cv_scores.std():.4f}        | {acc:.4f}    | {auc_:.4f}")


### 3.9 Gráfico 16 — Comparativa completa (Pacientes)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

nombres_pac = list(resumen_pac.keys())
f1m_vals  = [resumen_pac[n]['f1m']   for n in nombres_pac]
cv_mus_p  = [resumen_pac[n]['cv_mu'] for n in nombres_pac]
cv_sds_p  = [resumen_pac[n]['cv_sd'] for n in nombres_pac]
colors_p  = ['#FFCC80','#FFCC80','#FFCC80','#E65100','#7B1FA2']

ax = axes[0]
bars = ax.bar(nombres_pac, f1m_vals, color=colors_p, edgecolor='white', width=0.6)
for b, v in zip(bars, f1m_vals):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.005,
            f'{v:.3f}', ha='center', fontsize=9, fontweight='bold')
ax.set_ylim(0.4, 1.0)
ax.set_ylabel('F1-Macro (Test)', fontsize=11)
ax.set_title('F1-Macro en test — Pacientes\n(baseline en naranja claro · avanzados en color)',
             fontweight='bold')
ax.tick_params(axis='x', rotation=15)
ax.axhline(0.606, color='gray', linestyle='--', lw=1, alpha=0.6, label='Baseline mejor (0.606)')
ax.legend(fontsize=8)

ax = axes[1]
ax.bar(nombres_pac, cv_mus_p, yerr=cv_sds_p, color=colors_p, edgecolor='white',
       width=0.6, capsize=5, error_kw=dict(elinewidth=1.2, ecolor='#333'))
for i, (mu, sd) in enumerate(zip(cv_mus_p, cv_sds_p)):
    ax.text(i, mu + sd + 0.008, f'{mu:.3f}\n±{sd:.3f}',
            ha='center', fontsize=8, fontweight='bold')
ax.set_ylim(0.4, 1.0)
ax.set_ylabel('F1-Macro CV (5-fold)', fontsize=11)
ax.set_title('Validación cruzada (5-fold)\nPacientes', fontweight='bold')
ax.tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('g16_comparativa_pac.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Gráfico guardado: g16_comparativa_pac.png")


### 3.10 Gráfico 17 — Matrices de confusión (modelos avanzados, Pacientes)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, (nombre, modelo) in zip(axes, {'Random Forest': rf_pac_best, 'XGBoost': xgb_pac_best}.items()):
    y_pred = modelo.predict(X_te_p)
    cm = confusion_matrix(y_te_p, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', ax=ax, cbar=False,
                xticklabels=clases, yticklabels=clases)
    f1m = f1_score(y_te_p, y_pred, average='macro', zero_division=0)
    acc = accuracy_score(y_te_p, y_pred)
    ax.set_title(f'{nombre}\nF1-Macro={f1m:.3f}  Acc={acc:.3f}', fontweight='bold')
    ax.set_xlabel('Predicho')
    ax.set_ylabel('Real')
plt.suptitle('Matrices de confusión — Modelos avanzados (Pacientes)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('g17_cm_pac_avanzado.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Gráfico guardado: g17_cm_pac_avanzado.png")


### 3.11 Gráfico 18 — SHAP Pacientes (mejor modelo)

In [ ]:
# ── Determinar mejor modelo Frente 2 ──────────────────────────────────
mejor_nombre_pac = max(resumen_pac, key=lambda n: resumen_pac[n]['f1m'])
mejor_modelo_pac = todos_pac[mejor_nombre_pac]
print(f"Mejor modelo Frente 2: {mejor_nombre_pac}")
print(f"  F1-Macro Test = {resumen_pac[mejor_nombre_pac]['f1m']:.4f}")
print(f"  AUC-Macro Test = {resumen_pac[mejor_nombre_pac]['auc']:.4f}")

explainer_pac = shap.TreeExplainer(mejor_modelo_pac)
np.random.seed(42)
idx_s = np.random.choice(len(X_te_p), min(200, len(X_te_p)), replace=False)
shap_vals_pac = explainer_pac.shap_values(X_te_p[idx_s])

# RF multiclase: lista de 3 arrays; XGB: array 3D — normalizar a array 2D (media entre clases)
if isinstance(shap_vals_pac, list):
    sv_pac = np.mean(np.abs(np.stack(shap_vals_pac, axis=0)), axis=0)
elif shap_vals_pac.ndim == 3:
    sv_pac = np.mean(np.abs(shap_vals_pac), axis=2)
else:
    sv_pac = shap_vals_pac

# Top 15 features por importancia SHAP media
feat_names = feat_cols if isinstance(feat_cols, list) else list(feat_cols)
mean_shap_pac = np.abs(sv_pac).mean(axis=0)

# Acortar nombres largos de OHE para el gráfico
short_names = []
for f in feat_names:
    f2 = f.replace('_enc','').replace('lugar_procedencia_','lug_').replace('tipo_tratamiento_','trat_')
    f2 = f2.replace('motivo_ingreso_','mot_').replace('motivo_reingreso_','reing_')
    f2 = f2.replace('diagnostico_','diag_').replace('estado_emocional_paciente','est_emoc')
    short_names.append(f2)

top_idx = np.argsort(mean_shap_pac)[-15:]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Bar plot top-15
ax = axes[0]
colors_pac = plt.cm.Purples(np.linspace(0.4, 0.9, 15))
ax.barh([short_names[i] for i in top_idx], mean_shap_pac[top_idx], color=colors_pac)
ax.set_xlabel('Importancia media |SHAP|', fontsize=11)
ax.set_title(f'SHAP — Top 15 variables influyentes\n({mejor_nombre_pac}, Frente 2)',
             fontweight='bold', fontsize=12)

# Scatter
ax2 = axes[1]
for rank, i in enumerate(top_idx):
    vals = sv_pac[:, i]
    feat = X_te_p[idx_s, i]
    sc = ax2.scatter(vals, np.full_like(vals, rank) + np.random.normal(0, 0.08, len(vals)),
                     c=feat, cmap='coolwarm', alpha=0.5, s=12, vmin=0, vmax=1)
ax2.set_yticks(range(15))
ax2.set_yticklabels([short_names[i] for i in top_idx], fontsize=8)
ax2.axvline(0, color='gray', lw=0.8, linestyle='--')
ax2.set_xlabel('Valor SHAP (impacto en predicción)', fontsize=11)
ax2.set_title('Distribución SHAP por variable\n(azul=bajo · rojo=alto)',
              fontweight='bold', fontsize=12)
plt.colorbar(sc, ax=ax2, label='Valor normalizado')

plt.tight_layout()
plt.savefig('g18_shap_pacientes.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Gráfico guardado: g18_shap_pacientes.png")


---
## ✅ Resumen final Hito 3 — Comparativa completa

In [ ]:
print("=" * 75)
print("FRENTE 1 — INVENTARIO (Alerta 7 días)")
print("=" * 75)
print(f"{'Modelo':22s} | {'F1 Test':>8s} | {'AUC Test':>9s} | {'F1 CV μ':>8s} | {'F1 CV σ':>7s}")
print("-" * 65)
for n, v in resumen_inv7.items():
    marca = " ← MEJOR" if n == mejor_nombre_inv else ""
    print(f"{n:22s} | {v['f1_test']:.4f}   | {v['auc_test']:.4f}    | {v['cv_mu']:.4f}   | {v['cv_sd']:.4f}{marca}")

print()
print("=" * 75)
print("FRENTE 2 — PACIENTES (nivel de riesgo Bajo/Medio/Alto)")
print("=" * 75)
print(f"{'Modelo':22s} | {'Acc Test':>9s} | {'F1-Macro':>9s} | {'AUC-Macro':>9s} | {'F1 CV μ':>7s} | {'F1 CV σ':>7s}")
print("-" * 78)
for n, v in resumen_pac.items():
    marca = " ← MEJOR" if n == mejor_nombre_pac else ""
    print(f"{n:22s} | {v['acc']:.4f}    | {v['f1m']:.4f}    | {v['auc']:.4f}     | {v['cv_mu']:.4f}  | {v['cv_sd']:.4f}{marca}")

print()
print(f"Modelo final Frente 1: {mejor_nombre_inv}")
print(f"Modelo final Frente 2: {mejor_nombre_pac}")


### 3.12 Guardar modelos finales

In [ ]:
# ── Serializar mejores modelos en .pkl ────────────────────────────────
BASE = os.path.dirname(os.path.abspath('__file__')) if '__file__' in dir() else os.getcwd()

models_inv_final = {
    'mejor_nombre':  mejor_nombre_inv,
    'modelo_7d':     mejor_modelo_inv,
    'modelo_14d':    rf_inv14_best if mejor_nombre_inv == 'Random Forest' else xgb_inv14_best,
    'scaler':        scaler_inv,
    'le_categoria':  le_cat,
    # baseline (para retrocompatibilidad con el dashboard)
    'arbol_7d':      modelos_f1['Árbol (depth=5)'],
    'arbol_14d':     modelos_f1_14['Árbol (depth=5)'],
    'rf_7d':         rf_inv_best,
    'xgb_7d':        xgb_inv_best,
}

models_pac_final = {
    'mejor_nombre':  mejor_nombre_pac,
    'modelo':        mejor_modelo_pac,
    'scaler':        scaler_p,
    'le_target':     le_target,
    'feat_cols':     feat_cols,
    'clases':        clases,
    # baseline
    'arbol':         modelos_f2['Árbol (depth=5)'],
    'rf':            rf_pac_best,
    'xgb':           xgb_pac_best,
}

joblib.dump(models_inv_final, os.path.join(BASE, 'models_inventario.pkl'))
joblib.dump(models_pac_final, os.path.join(BASE, 'models_pacientes.pkl'))

print("✓ Modelos serializados:")
print(f"  models_inventario.pkl — Frente 1 ({mejor_nombre_inv})")
print(f"  models_pacientes.pkl  — Frente 2 ({mejor_nombre_pac})")
